# 12 — Forecast chat with RAG

Ask a vacancy forecast question using a province and an occupation or industry from the saved catalog. The router selects the series and 1Q/2Q/4Q horizon; unclear or unsupported requests return a clarification before inference.

`ForecastService` reuses the pinned adapter for numbers and the base model with dated bulletin passages for a cited explanation. It loads saved model/index assets on demand and keeps the forecast numbers separate from the explanation. Upload the updated `jobai` folder with this notebook.

Run setup once. Edit and rerun the last cell to ask another question; `CHAT_CONTEXT` carries the selected series and horizon into follow-ups such as “What about six months?” Set `CHAT_CONTEXT = None` to start a new conversation.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
except ImportError:
    pass

# Colab supplies CUDA-enabled PyTorch. Install runtime packages before importing transformers.
%pip install -q chromadb sentence-transformers pyyaml pandas "transformers==5.17.0" "peft==0.20.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" "safetensors==0.8.0" "huggingface-hub==1.31.0"

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

REPO = Path.cwd()
sys.path.insert(0, str(REPO))
from jobai.chat import ForecastService

ALLOW_EMBEDDING_DOWNLOAD = False  # Enable once only if embedding weights are absent from both caches.
service = ForecastService(REPO, allow_embedding_download=ALLOW_EMBEDDING_DOWNLOAD)
CHAT_CONTEXT = None
print("Pinned model:", service.run["run_id"])

In [ ]:
# Change this question and rerun this cell; the service and model stay loaded.
USER_QUESTION = "What is the vacancy outlook for civil engineers in Uusimaa?"

result = service.answer_question(USER_QUESTION, context=CHAT_CONTEXT)
CHAT_CONTEXT = result["context"]
display(Markdown(result["answer"]))
if result["status"] == "answered":
    forecasts = pd.DataFrame(result["forecasts"])
    display(forecasts[["origin_quarter", "target_quarter", "horizon_q", "last_value", "y_pred"]].round(2))
    print("Selected series:", result["request"]["series_id"])
    print(f"RAG status: {result['rag']['status']} | dated passages: {len(result['rag']['passages'])}")
    (REPO / "reports").mkdir(exist_ok=True)
    forecasts.to_csv(REPO / "reports/rag_forecast_demo_predictions.csv", index=False)
    _ = (REPO / "reports/rag_forecast_demo_answer.md").write_text(result["answer"], encoding="utf-8")
    _ = (REPO / "reports/rag_forecast_demo_evidence.json").write_text(
        json.dumps(result["rag"], ensure_ascii=False, indent=2), encoding="utf-8")
elif result["request"]["choices"]:
    display(pd.DataFrame(result["request"]["choices"]))